# Stablecoin Depeg Monitor

**The question:** How often have USDC, USDT, and DAI lost their $1 peg since 2021, how badly, and which one held up best?

**Who's asking:** a treasury team holding millions in stablecoins that needs to know which one is safest to hold.

**The approach (monitoring):**
1. Define normal: a price close to $1.
2. Define a problem: the daily low below $0.99 or the daily high above $1.01.
3. Scan every day for a break of that rule.
4. Group consecutive flagged days into events and rank them by how bad they were.
5. Explain the worst ones.

## Step 1: Get the prices

Depegs often happen and recover within hours, so the closing price alone can hide them. If USDC fell to $0.88 at 3am and recovered to $0.999 by midnight, the close would show $0.999 but the **low** would show $0.88. So I pull the daily **low** and **high** as well as the close, for every day from January 2021 to September 2026.

In [1]:
import yfinance as yf
import pandas as pd

coins = ["USDC-USD", "USDT-USD", "DAI-USD"]
data = yf.download(coins, start="2021-01-01", end="2026-10-01", progress=False)
data.to_csv("../data/raw_prices.csv")

low, high, close = data["Low"].copy(), data["High"].copy(), data["Close"].copy()
print(f"{len(data)} days, {data.index.min().date()} to {data.index.max().date()}")

2099 days, 2021-01-01 to 2026-09-30


## Step 2: Check the data

In a monitoring project a single bad price looks exactly like a real emergency, so the data has to be checked before anything gets flagged. First: missing days, the most extreme prices, and a summary of each coin.

In [2]:
print("Missing days per coin:")
print(close.isna().sum(), "\n")
print("Lowest daily low:")
print(low.min(), "\n")
print("Highest daily high:")
print(high.max())

Missing days per coin:
Ticker
DAI-USD     0
USDC-USD    0
USDT-USD    0
dtype: int64 

Lowest daily low:
Ticker
DAI-USD     0.897003
USDC-USD    0.877400
USDT-USD    0.948486
dtype: float64 

Highest daily high:
Ticker
DAI-USD     3.668398
USDC-USD    2.349556
USDT-USD    1.033019
dtype: float64


In [3]:
close.describe()

Ticker,DAI-USD,USDC-USD,USDT-USD
count,2099.000000,2099.000000,2099.000000
mean,1.000040,0.999980,1.000056
std,0.001080,0.000746,0.000727
min,0.973869,0.971500,0.995872
25%,0.999737,0.999865,0.999751
50%,0.999925,0.999982,1.000094
75%,1.000142,1.000085,1.000343
max,1.010310,1.010496,1.011530


No days are missing. But a USDC high of about $2.35 and a DAI high of about $3.67 are impossible for a stablecoin. Here are the 5 lowest lows and 5 highest highs for each coin:

In [4]:
for c in coins:
    print(c)
    print("  lowest lows:  ", {str(d.date()): round(v, 4) for d, v in low[c].nsmallest(5).items()})
    print("  highest highs:", {str(d.date()): round(v, 4) for d, v in high[c].nlargest(5).items()})

USDC-USD
  lowest lows:   {'2023-03-11': 0.8774, '2023-03-12': 0.947, '2021-09-09': 0.9874, '2023-03-13': 0.988, '2023-03-21': 0.9964}
  highest highs: {'2021-11-16': 2.3496, '2021-11-17': 1.1735, '2021-04-18': 1.0265, '2022-05-12': 1.0114, '2021-04-04': 1.0111}
USDT-USD
  lowest lows:   {'2022-05-12': 0.9485, '2022-11-10': 0.9815, '2026-05-07': 0.9896, '2026-05-08': 0.9896, '2022-05-11': 0.9935}
  highest highs: {'2021-11-16': 1.033, '2023-03-11': 1.0296, '2021-04-18': 1.028, '2021-11-17': 1.02, '2023-03-12': 1.0163}
DAI-USD
  lowest lows:   {'2023-03-11': 0.897, '2023-03-12': 0.9498, '2021-03-18': 0.9798, '2021-11-10': 0.9846, '2021-12-04': 0.986}
  highest highs: {'2021-11-16': 3.6684, '2021-11-17': 1.5008, '2021-04-18': 1.0266, '2021-03-29': 1.0234, '2021-05-19': 1.0199}


### Cross-checking against a second source

The guide suggests checking suspicious dates on CoinGecko, but CoinGecko's free API only returns the last 365 days, and its daily snapshot doesn't include intraday lows and highs anyway. Instead I download daily low and high prices from two exchanges with free public APIs:

- **Coinbase** for DAI (until Coinbase delisted DAI in May 2026) and USDT (from May 2021)
- **Bitstamp** for USDC and USDT (USDT from June 2021)

**My rule for a bad record:** a price that crosses $0.99 or $1.01 on Yahoo Finance is kept only if the second source *also* crossed that line on the same day. If the second source stayed inside the band, the Yahoo price is a bad record. If there is no second-source data for that day, I keep the price and mark it as unverified.

In [5]:
import requests, time

def coinbase_daily(product, start="2021-01-01", end="2026-10-01"):
    rows, s, end = [], pd.Timestamp(start), pd.Timestamp(end)
    while s < end:
        e = min(s + pd.Timedelta(days=299), end)       # Coinbase returns at most 300 candles per request
        url = f"https://api.exchange.coinbase.com/products/{product}/candles"
        r = requests.get(url, params={"granularity": 86400, "start": s.isoformat(), "end": e.isoformat()}, timeout=30)
        rows += r.json()
        s = e
        time.sleep(0.4)
    df = pd.DataFrame(rows, columns=["time", "low", "high", "open", "close", "volume"]).drop_duplicates("time")
    df.index = pd.to_datetime(df["time"], unit="s")
    return df.sort_index()[["low", "high"]]

def bitstamp_daily(pair, start="2021-01-01", end="2026-10-01"):
    rows, ts = [], int(pd.Timestamp(start).timestamp())
    while ts < pd.Timestamp(end).timestamp():
        url = f"https://www.bitstamp.net/api/v2/ohlc/{pair}/"
        batch = requests.get(url, params={"step": 86400, "limit": 1000, "start": ts}, timeout=30).json()["data"]["ohlc"]
        if not batch:
            break
        rows += batch
        ts = int(batch[-1]["timestamp"]) + 86400
        time.sleep(0.4)
    df = pd.DataFrame(rows)
    df.index = pd.to_datetime(df["timestamp"].astype(int), unit="s")
    return df[["low", "high"]].astype(float)

cb_dai, cb_usdt = coinbase_daily("DAI-USD"), coinbase_daily("USDT-USD")
bs_usdc, bs_usdt = bitstamp_daily("usdcusd"), bitstamp_daily("usdtusd")

# For USDT there are two exchanges: take the lowest low and highest high across them
check_low = pd.DataFrame({
    "USDC-USD": bs_usdc["low"],
    "USDT-USD": pd.concat([cb_usdt["low"], bs_usdt["low"]], axis=1).min(axis=1),
    "DAI-USD": cb_dai["low"],
}).reindex(low.index)
check_high = pd.DataFrame({
    "USDC-USD": bs_usdc["high"],
    "USDT-USD": pd.concat([cb_usdt["high"], bs_usdt["high"]], axis=1).max(axis=1),
    "DAI-USD": cb_dai["high"],
}).reindex(high.index)

pd.concat({"low": check_low, "high": check_high}, axis=1).to_csv("../data/crosscheck_prices.csv")
print("Days with second-source data:")
print(check_low.notna().sum())

Days with second-source data:
USDC-USD    2099
USDT-USD    1976
DAI-USD     1950
dtype: int64


Now apply the rule. A bad low or high gets replaced with the second source's price for that day, so the day stays in the data but the bad spike doesn't.

In [6]:
bad_low = (low < 0.99) & check_low.notna() & (check_low >= 0.99)
bad_high = (high > 1.01) & check_high.notna() & (check_high <= 1.01)
unverified = ((low < 0.99) | (high > 1.01)) & check_low.isna()

def as_table(mask, yahoo, second, side):
    t = pd.DataFrame({"yahoo": yahoo.where(mask).stack().dropna(),
                      "second_source": second.where(mask).stack().dropna()})
    return t.assign(side=side).round(4)

removed = pd.concat([as_table(bad_low, low, check_low, "low"),
                     as_table(bad_high, high, check_high, "high")]).sort_index()
removed.index.names = ["date", "coin"]
print(f"Removed {len(removed)} bad prices on {removed.index.get_level_values('date').nunique()} days")
print(removed.groupby("coin").size(), "\n")
print("Flagged days with no second source (kept, unverified):")
print(unverified.stack()[lambda s: s])

low = low.mask(bad_low, check_low)
high = high.mask(bad_high, check_high)
removed

Removed 34 bad prices on 25 days
coin
DAI-USD     23
USDC-USD     6
USDT-USD     5
dtype: int64 

Flagged days with no second source (kept, unverified):
Date        Ticker  
2021-04-04  USDT-USD    True
2021-04-17  USDT-USD    True
2021-04-18  USDT-USD    True
dtype: bool


yahoo  second_source  side
date       coin                                 
2021-03-18 DAI-USD   0.9798         1.0006   low
2021-03-29 DAI-USD   1.0234         1.0014  high
2021-04-02 DAI-USD   1.0103         1.0011  high
2021-04-03 DAI-USD   1.0111         1.0013  high
2021-04-04 DAI-USD   1.0129         1.0012  high
           USDC-USD  1.0111         1.0041  high
2021-04-17 DAI-USD   1.0113         1.0013  high
2021-04-18 DAI-USD   1.0266         1.0076  high
2021-05-19 DAI-USD   0.9896         0.9970   low
2021-06-15 DAI-USD   1.0114         1.0018  high
           USDC-USD  1.0102         1.0099  high
           USDT-USD  1.0104         1.0026  high
2021-09-09 USDC-USD  0.9874         0.9999   low
2021-11-10 DAI-USD   0.9846         0.9990   low
           DAI-USD   1.0198         1.0005  high
2021-11-11 DAI-USD   0.9881         0.9994   low
2021-11-12 DAI-USD   1.0124         1.0010  high
2021-11-16 DAI-USD   3.6684         1.0008  high
           USDC-USD  2.3496         1.0010  high
           USDT-USD  1.0330         1.0065  high
2021-11-17 DAI-USD   1.5008         1.0004  high
           USDC-USD  1.1735         1.0015  high
           USDT-USD  1.0200         1.0037  high
2021-12-04 DAI-USD   0.9860         0.9995   low
2021-12-06 DAI-USD   1.0111         1.0006  high
2021-12-07 DAI-USD   1.0105         1.0004  high
2021-12-13 DAI-USD   0.9896         0.9994   low
2021-12-15 DAI-USD   1.0103         1.0003  high
2021-12-17 DAI-USD   0.9899         0.9992   low
2022-05-12 DAI-USD   1.0138         1.0010  high
           USDC-USD  1.0114         1.0011  high
2025-01-01 DAI-USD   1.0117         1.0100  high
2026-05-07 USDT-USD  0.9896         0.9980   low
2026-05-08 USDT-USD  0.9896         0.9977   low

### Data checks

- **Missing days:** none. All 2,099 days from Jan 1, 2021 to Sep 30, 2026 are present for all three coins.
- **Impossible prices:** on Nov 16 and 17, 2021 Yahoo shows highs of $2.35 for USDC, $3.67 and $1.50 for DAI, and $1.03 for USDT, on the same days, with no news. Both exchanges show all three coins within a fraction of a cent of $1. These are bad records and were removed.
- **Bad lows that would have looked like depegs:** USDT at $0.9896 on May 7 and 8, 2026 (Coinbase low $0.998, Bitstamp $0.9997, and CoinGecko's hourly prices never went below $0.9997), USDC at $0.9874 on Sep 9, 2021, and seven DAI lows between $0.980 and $0.990 in 2021 that Coinbase doesn't show. All removed.
- **In total I removed 34 prices on 25 days** (USDC 6, USDT 5, DAI 23). Most are small highs just above $1.01 in 2021 that the exchanges don't confirm.
- **Kept:** every price from the events that matter. USDC and DAI in March 2023, USDT in May 2022 (Terra collapse) and November 2022 (FTX collapse) are all confirmed by the second source, which in each case shows an equal or deeper move.
- **Unverified:** three USDT highs in April 2021 (the 4th, 17th, and 18th) happened before either exchange listed USDT. I kept them. USDC and DAI also spiked above $1.01 on April 18, 2021 on the exchanges, which makes them plausible.
- **Closing prices:** a few 2021 DAI closes sit more than half a cent from $1 where Coinbase shows a tighter range (for example $0.9878 on Nov 10, 2021). Closes only feed the day-to-day stability measures, not the depeg flags, so I left them in. If anything, this makes DAI look slightly less stable than it was.